# ODSC 2026 — Notebook 1: Custom Claw

Compliance is a real issue for virtually every company. Entire departments exist for the sole purpose of ensuring compliance is up to date for every vertical in every region they do business. Here we'll build a compliance monitor from scratch — focused on teacher licensure for educators across the United States.

We'll build a **pydantic-ai agent** following the Principle of Least Privilege (PoLP): the agent only gets the capabilities it needs, added one at a time.

| Part | What you build |
| --- | --- |
| **Part 1** | Environment setup |
| **Part 2** | Custom Claw — raw LLM → add skills → pydantic-ai agent |
| **Part 3** | Extend with Brave Search *(optional)* |
| **Part 3b** | Minimum viable conversational agent *(optional)* |
| **Part 4** | Autonomous scheduling — run compliance checks on a cron loop |

**Up next:** `2_openclaw.ipynb` — the same logic in a production agent framework.

---
## Part 1: Environment Setup

In [ ]:
import subprocess
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
BRAVE_API_KEY  = os.getenv("BRAVE_API_KEY", "")

assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"

if not BRAVE_API_KEY or BRAVE_API_KEY.startswith("your-"):
    print("⚠️  BRAVE_API_KEY not set — Parts 3 and 3b will be skipped")
    BRAVE_API_KEY = ""
else:
    print("✅ Brave key set")

docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

print("✅ OpenAI key set")
print("✅ Docker running")
print("\nReady to go.")

---
## Part 2: From LLM to Agent

A raw LLM knows a lot from training — but it can't browse live pages, remember anything between runs, or take action in the world. An **agent** can.

The difference is skills: Python functions the LLM can call. We'll add them one at a time so you can see exactly where the capability comes from.

**Architecture:** `agent.py` is ~60 lines. Four skills in `src/skills/`. The agent decides when and how to call each one — you don't write the routing logic.

### Step 1: The Base LLM

Just an OpenAI call. It knows a lot from training, but it can't browse live pages or remember anything between runs. This is a good baseline — but not something we can trust for up-to-date information or rely on to maintain a database of crucial compliance data.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What Praxis tests are required for Math 7-12 in Ohio?"}]
)
print(response.choices[0].message.content)
# Gives training-data knowledge — can't verify today's Ohio DOE page.

### Step 2: Add a Skill

Decorate any Python function with `agent.tool_plain()` and the LLM can call it. **One line turns a function into a skill.** And the best part is you can ask the LLM to generate the function for you! 

In [ ]:
import sys; sys.path.insert(0, "custom-claw/src")

from pydantic_ai import Agent
from skills.fetch_page import fetch_compliance_page

# Show what the skill returns directly — so it's clear the tool ran
raw = fetch_compliance_page("https://education.ohio.gov/Topics/Teaching/Licensure")
print(f"Skill fetched {len(raw):,} chars. Preview:\n{raw[:300]}...\n{'─'*60}\n")

# Now give that skill to an agent
agent = Agent(
    model="openai:gpt-4o-mini",
    instructions=(
        "You monitor teacher licensure compliance pages. "
        "When you fetch a page, always describe what content you retrieved "
        "and what it says about certification requirements — even if the page "
        "is sparse or navigation-heavy, report exactly what was there."
    )
)

agent.tool_plain(fetch_compliance_page)  # ← one line = one new skill

result = await agent.run(
    "Fetch https://education.ohio.gov/Topics/Teaching/Licensure "
    "and tell me what the page says about teacher certification requirements."
)
print(result.output)

**Every skill is just a Python function** and you don't even need to write it yourself. The same LLM you're using as your agent can write its own skills. No separate window, no switching tools. Let's see it do that now.

In [ ]:
# Ask the base LLM to write the fetch skill for us
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": (
            "Write a Python function called fetch_compliance_page(url: str) -> str "
            "that fetches a web page and returns its cleaned text content. "
            "Strip all HTML tags, script blocks, and style blocks. "
            "Collapse whitespace. Include a one-line docstring. "
            "Return only the function — no explanation."
        )
    }]
)
print(response.choices[0].message.content)
# This is essentially what lives in custom-claw/src/skills/fetch_page.py
# We added SSRF protection on top — but the core logic the LLM just wrote.

### Step 3: Add More Skills

Each new capability is one function. The agent decides when and how to call each one, you don't write the routing logic. 

In [ ]:
from skills.check_change import check_for_change
from skills.discover_urls import discover_urls

# Open custom-claw/src/skills/ — each file is one skill.
# Adding a new one: write a function, register it below.

agent.tool_plain(check_for_change)  # compare page content to stored snapshot
agent.tool_plain(discover_urls)     # Brave Search for new compliance pages

### Step 4: Natural Language In, Compliance Check Out
Let's test it out! 

In [ ]:
from agent import run  # imports the fully-configured agent

result = await run("Check Ohio and Texas for any teacher licensure changes and alert me if anything changed.")
print(result)

### The Ceiling

To get here we wrote:
- SSRF-safe URL fetching with IP pinning
- SHA-256 content hashing + SQLite snapshot diffing  
- A tool-routing loop the agent orchestrates

**Adding a new skill is one file + one line. That's genuinely easy.**

What isn't easy: owning all the plumbing above forever. Networking, storage, error handling, auth, scheduling — you wrote it, you maintain it.

> **Up next:** OpenClaw does all of this from a single chat message. No plumbing required.

---
## Part 3: Extending Custom Claw with Brave Search *(Optional)*

**Skip this part if you don't have a Brave API key** — free at [brave.com/search/api](https://brave.com/search/api/) but not required for the rest of the workshop.

### The limitation we're solving

Custom Claw's watch list is hardcoded in `compliance_urls.py`. If a state adds a new page, or we want to monitor a district we've never seen before, we have to edit the code.

What if the agent could *find* the right URLs itself?

### The idea

Add a `discover_urls(state)` function that:
1. Queries Brave Search for `"{state} teacher licensure certification requirements site:.gov"`
2. Returns the top results as candidate URLs
3. Feeds them into the existing hash/diff/alert flow

This is exactly what OpenClaw's `brave-search` skill does automatically when you ask it to monitor a new state — no code changes needed. We're building a manual version of that here to show the concept.

In [ ]:
import os
import requests
from dotenv import find_dotenv, dotenv_values

_env = dotenv_values(find_dotenv())

def discover_compliance_urls(state: str, max_results: int = 3) -> list[dict]:
    api_key = _env.get("BRAVE_API_KEY", "")
    if not api_key:
        print("⚠️  BRAVE_API_KEY not set — skipping URL discovery")
        return []

    query = f"{state} department of education teacher licensure certification requirements"
    resp = requests.get(
        "https://api.search.brave.com/res/v1/web/search",
        headers={"Accept": "application/json", "X-Subscription-Token": api_key},
        params={"q": query, "count": max_results},
        timeout=10,
    )
    resp.raise_for_status()
    return [
        {"title": r["title"], "url": r["url"]}
        for r in resp.json().get("web", {}).get("results", [])
    ]


state = "Indiana"
print(f"Discovering compliance URLs for {state}...\n")
for r in discover_compliance_urls(state):
    print(f"  {r['title']}\n  {r['url']}\n")

---
### Part 3b: Minimum Viable Conversational Agent *(Optional)*

We added Brave Search, but it's still structured code, you have to call `discover_compliance_urls("Indiana")` yourself.

What if you could just *ask* it?

Below is a ~50-line conversational wrapper around the functions you've already built. It uses the OpenAI API to interpret a natural language instruction, extract intent, and call the right function. This is the minimum viable version of what OpenClaw does under the hood.

> **Notice what you have to build yourself:** the system prompt (your mini `AGENTS.md`), the intent parser, the tool routing, the response formatter. OpenClaw handles all of this for you, the gap between this cell and Part 5 is everything a production agent framework provides.

In [ ]:
import json, asyncio
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a teacher licensure compliance monitoring agent.

You have two capabilities:
1. check_compliance(states: list[str]) — check monitored URLs for changes in specific states
2. discover_urls(state: str) — search for new compliance pages for a state

When the user gives you an instruction, respond with a JSON object like:
  {"action": "check_compliance", "states": ["Ohio", "Texas"]}
  {"action": "discover_urls", "state": "Indiana"}
  {"action": "unknown", "message": "I can only check compliance or discover URLs"}

Respond with JSON only. No explanation."""

def parse_intent(user_message: str) -> dict:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

async def run_agent(user_message: str):
    print(f"You: {user_message}")
    intent = parse_intent(user_message)
    action = intent.get("action")

    if action == "check_compliance":
        states = intent.get("states", [])
        print(f"Agent: Running compliance check for {', '.join(states)}...\n")
        # Delegate to the pydantic-ai agent we built in Part 2
        import sys; sys.path.insert(0, "custom-claw/src")
        from agent import run
        result = await run(f"Check {', '.join(states)} for any teacher licensure changes.")
        print(f"Agent: {result}")

    elif action == "discover_urls":
        state = intent.get("state", "")
        print(f"Agent: Searching for compliance pages for {state}...\n")
        results = discover_compliance_urls(state)
        if results:
            print(f"Agent: Found {len(results)} page(s) for {state}:")
            for r in results:
                print(f"  • {r['title']}")
                print(f"    {r['url']}")
        else:
            print("Agent: No results found (Brave API not configured).")

    else:
        print(f"Agent: {intent.get('message', 'I can check compliance or discover URLs for a state.')}")
    print()


# --- Try it ---
# Notice what you're manually writing: the system prompt (your mini AGENTS.md),
# the intent parser, the routing logic, the response formatter.
# OpenClaw handles all of this — the gap between this cell and Notebook 2 is
# everything a production agent framework provides.
await run_agent("Check for any licensure changes in Ohio and Texas")
await run_agent("Find compliance pages for Indiana")
await run_agent("What's the weather like?")

---
## Part 4: Autonomous Scheduling

The agent works — but right now you have to run it manually every time. That's not compliance monitoring, that's compliance checking.

To make it autonomous, we add one thing: a loop with a sleep. Since Custom Claw is already async, `asyncio.sleep()` is all we need — no new dependencies.

> **The honest trade-off:** This cell blocks while it runs, and the monitoring stops the moment you close the notebook or shut down the kernel. That's the ceiling of a notebook agent.
>
> This is *exactly* the gap that OpenClaw closes. OpenClaw runs in Docker — always on, survives restarts, no notebook required. The code below is the reason Part 2 exists: so you understand what you're handing off when you move to a framework.

In [ ]:
import asyncio
import datetime
import sys
sys.path.insert(0, "custom-claw/src")
from agent import run

INTERVAL_HOURS = 1  # set to 1/12 for every 5 minutes during demos

async def compliance_monitor():
    while True:
        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
        print(f"[{now}] Running scheduled compliance check...\n")

        result = await run(
            "Check all monitored states for any teacher licensure or Praxis requirement changes. "
            "Report what changed and which state it affects. If nothing changed, say so briefly."
        )
        print(result)

        next_run = datetime.datetime.now() + datetime.timedelta(hours=INTERVAL_HOURS)
        print(f"\n── Next check at {next_run.strftime('%H:%M')} ──\n")
        await asyncio.sleep(INTERVAL_HOURS * 3600)

print(f"Compliance monitor running every {INTERVAL_HOURS}h. Interrupt (■) to stop.\n")
await compliance_monitor()